In [1]:
import os, sys

nb_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(nb_dir, '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
print('Using project root:', project_root)
print('First sys.path entry:', sys.path[0])

Using project root: /Users/rithvik/Documents/hnrs/Decoder
First sys.path entry: /Users/rithvik/Documents/hnrs/Decoder


In [2]:
import numpy as np
from scipy.sparse import csr_matrix, eye, hstack, save_npz, load_npz
import scipy.io

In [3]:
from utils.LDPC_encode import LDPCEncode
from utils.awgn_channel import AWGNChannel
from utils.find_ber import findBER
from utils.res_cluster_picker import pick_max_avg_residual_cluster, pick_max_max_residual_cluster

In [4]:
from ldpc.bp_decoder import BpDecoder

In [5]:
H_mat_dat = scipy.io.loadmat('H.mat')
H = csr_matrix(H_mat_dat['H'])

In [6]:
decoder = BpDecoder(H)

In [7]:
n = 486 # length of message
n_frames = 10000
max_iter = 30

message = np.random.randint(0, 2, (n_frames, n))
print("Message shape:", message.shape)

Message shape: (10000, 486)


In [8]:
encoded_codeword = LDPCEncode(message)
print("Encoded codeword shape:", encoded_codeword.shape) 

tx_codeword = 1 - 2 * encoded_codeword 

Encoded codeword shape: (10000, 648)


In [9]:
m, _ = H.shape
arr = np.arange(m)
clusters = arr.reshape(6, -1)


In [10]:
snrs = [-3, -2, -1, 0, 1, 2, 3, 4]
bers = []


for snr in snrs:
    rx_llrs = AWGNChannel(tx_codeword, snr_db=snr)
    decoded_codewords = []

    for i in range(n_frames):
        llr = rx_llrs[i, :]

        decoder.reset()
        decoder.initialise_log_domain_bp(llr)
        
        for iter in range(max_iter):
            residuals = decoder.get_residuals()
            cluster_idx, scheduled_cluster = pick_max_avg_residual_cluster(residuals, clusters)

            print(f"Scheduled cluster at iteration {iter}: {cluster_idx}")

            decoder.decode_cluster(scheduled_cluster)

        
        decoded_codeword = (llr < 0).astype(int)
        decoded_codewords.append(decoded_codeword)
        
    decoded_codewords = np.array(decoded_codewords)
    decoded_message = decoded_codewords[:, :n]
    ber = findBER(message, decoded_message)
    bers.append(ber)
    print(f"BER at SNR {snr} dB: {ber}")

BER at SNR -3 dB: 0.15817510288065845
BER at SNR -2 dB: 0.13059238683127572
BER at SNR -1 dB: 0.10386975308641976
BER at SNR 0 dB: 0.07887654320987654
BER at SNR 1 dB: 0.056094650205761314
BER at SNR 2 dB: 0.037488477366255143
BER at SNR 3 dB: 0.022904115226337448
BER at SNR 4 dB: 0.012455555555555555
